# Embed WCAG Skill into Qwen3.5-0.8B

Closes the skill-to-weights loop: vegaml fitted on 300 WCAG examples writes the decision valleys into Qwen3.5-0.8B.

Method: closed-form MEMIT-style column edit on layer 23 MLP down_proj, 256 columns, 4 doses.

Reference: embedding.in blog post + fast-edits-decsion-model notebook.

In [ ]:
QUICK = False
N_FIT = 300
N_VAL = 40
LAYER = 23
WIDTH = 256
LAMBDAS = [0.3, 1.0, 3.0]
ROUNDS = 6
DOSES = [0.1, 0.25, 0.5, 1.0]
GUARD_WEIGHT = 1.0
GUARD_CAP = 0.03
SEED = 0
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

WCAG_QUESTIONS = {
    "wcag_violation": {
        "type": "boolean",
        "instructions": "Does this HTML/CSS markup fail any WCAG 2.2 criterion?"
    },
}

def wcag_prompt(state):
    return "Analyze this HTML/CSS snippet for WCAG 2.2 violations.\n\n" + state + "\n\n" + "Question: Does this markup have a WCAG 2.2 violation?\nAnswer (yes or no): "

print("Config loaded")
print("N_FIT=%d, N_VAL=%d, LAYER=%d, WIDTH=%d" % (N_FIT, N_VAL, LAYER, WIDTH))
print("MODEL: %s" % MODEL_NAME)


In [ ]:
import copy, glob, json, math, os, random, sys, time
from collections import Counter
from typing import Any, Dict, List, Optional, Sequence, Tuple
import numpy as np
import torch
import torch.nn.functional as F

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())


In [ ]:
!pip install -q vegaml
import vegaml
print("vegaml", vegaml.__version__, "installed and imported")


In [ ]:
# Embedding 360 WCAG training examples (base64-encoded to avoid escaping issues)
import base64
_B64 = "..."
rows = [json.loads(line) for line in base64.b64decode(_B64).decode().strip().split("\n") if line.strip()]
print("Loaded", len(rows), "WCAG examples (embedded inline)")
print("Loading vegaml...")
v = vegaml.load()
print("vegaml", vegaml.__version__, "loaded")

random.seed(SEED)
random.shuffle(rows)
fit_rows = rows[:N_FIT]
val_rows = rows[N_FIT:N_FIT+N_VAL]
test_rows = rows[N_FIT+N_VAL:]
print("Split: fit=%d, val=%d, test=%d" % (len(fit_rows), len(val_rows), len(test_rows)))

print("Fitting vegaml policy...")
t0 = time.time()
policy = vegaml.fit_policy(v, fit_rows, WCAG_QUESTIONS)
print("Policy fitted in %.1fs" % (time.time()-t0))

correct = sum(1 for r in val_rows if policy.decide(r["state"])["answers"]["wcag_violation"]["choice"] == ("true" if r["gold"]["wcag_violation"] else "false"))
val_acc = correct/len(val_rows) if val_rows else 0
print("Val accuracy: %.1f%% (%d/%d)" % (val_acc*100, correct, len(val_rows)))


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
student = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map="cuda:0")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Student: %d params" % sum(p.numel() for p in student.parameters()))

target_layer = student.model.layers[LAYER]
down_proj = target_layer.mlp.down_proj
print("Target: layer %d, down_proj shape %s" % (LAYER, down_proj.weight.shape))

yes_id = tokenizer("yes").input_ids[0]
no_id = tokenizer("no").input_ids[0]
print("yes=%d, no=%d" % (yes_id, no_id))


In [ ]:
def encode(rows_list):
    encs = []
    for r in rows_list:
        tokens = tokenizer(wcag_prompt(r["state"]), return_tensors="pt").input_ids.to(student.device)
        encs.append(tokens.shape[1]-1)
    return encs

train_encs = encode(fit_rows)
val_encs = encode(val_rows)
test_encs = encode(test_rows)
print("Encoded: train=%d, val=%d, test=%d" % (len(train_encs), len(val_encs), len(test_encs)))

teacher_gaps = []
for r in fit_rows:
    out = policy.decide(r["state"])
    p = out["answers"]["wcag_violation"]["boolean"]
    teacher_gaps.append(math.log(max(p, 1e-7)/max(1-p, 1e-7)))
teacher_gaps = torch.tensor(teacher_gaps, dtype=torch.float32, device=student.device)

print("Extracting keys...")
keys = []
for i, r in enumerate(fit_rows):
    tokens = tokenizer(wcag_prompt(r["state"]), return_tensors="pt").input_ids.to(student.device)
    k_tensor = [None]
    handle = down_proj.register_forward_pre_hook(lambda mod, inp: k_tensor.__setitem__(0, inp[0]))
    with torch.no_grad():
        student(input_ids=tokens, use_cache=False)
    handle.remove()
    if k_tensor[0] is not None:
        keys.append(k_tensor[0][0, train_encs[i], :].cpu())
    else:
        keys.append(torch.zeros(down_proj.in_features))
K = torch.stack(keys).to(student.device).float()
print("Keys:", K.shape)

print("Reading student gaps...")
student_gaps = []
for i, r in enumerate(fit_rows):
    tokens = tokenizer(wcag_prompt(r["state"]), return_tensors="pt").input_ids.to(student.device)
    with torch.no_grad():
        out = student(input_ids=tokens, use_cache=False)
        logits = out.logits[0, train_encs[i], :]
        student_gaps.append(logits[yes_id].item() - logits[no_id].item())
student_gaps = torch.tensor(student_gaps, dtype=torch.float32, device=student.device)

# R_i = desired change in down_proj output for row i
# down_proj output is added residually to hidden state, then goes through final_norm + lm_head
# For the two-option case: d_logit = teacher_gap - student_gap
# We want to shift the down_proj output to increase the yes/no logit gap by d_logit
# Simple approach: R = K * scale (scales the key by the gap magnitude)
# This is a first-order approximation
# V (values): desired change to down_proj output [300, 1024]
# down_proj: input [*, 3584] -> output [*, 1024] (added to residual)
# We need V such that down_proj(K^T) ≈ V^T, i.e., dW = V^T K (K^T K + λI)^{-1}
# The teacher gives us the desired yes/no logit gap change
# Map this to down_proj output space: use lm_head weight to project logit gap to residual
lm_head = student.lm_head.weight  # [vocab, 1024] for Qwen? No, Qwen ties weights differently
# Qwen uses weight tying: lm_head is the embedding matrix [vocab, hidden]
# The residual dim is 1024 (model.config.hidden_size)
# Simple approach: use a random projection from logit gap to 1024-d
# But we know yes_id and no_id, so we can use their embedding difference
yes_vec = student.model.embed_tokens.weight[yes_id]  # [1024]
no_vec = student.model.embed_tokens.weight[no_id]    # [1024]
direction = (yes_vec - no_vec).float()               # [1024] direction for "yes"
direction = direction / direction.norm()              # unit vector

scale = (teacher_gaps - student_gaps).float()  # [300]
V = direction.unsqueeze(0) * scale.unsqueeze(1)      # [300, 1024] - desired output change

# K is [300, 3584], V is [300, 1024]
# dW = V^T @ K @ (K^T @ K + λI)^{-1} = [1024, 300] @ [300, 3584] @ [3584, 3584] = [1024, 3584]
print("V shape:", V.shape, "| K shape:", K.shape)


In [ ]:
print("Solving...")
original_weight = down_proj.weight.clone()  # save pre-edit
best = None
best_acc = 0.0
for lam in LAMBDAS:
    KtK = K.T.float() @ K.float()
    A = KtK + lam * len(K) * KtK.diagonal().unsqueeze(0) * torch.eye(K.shape[1], device=student.device)
    A = A + 1e-6 * A.diagonal().mean() * torch.eye(K.shape[1], device=student.device)
    t0 = time.time()
    dW = torch.linalg.solve(A.float(), (K.T @ V).float()).T.contiguous().float()
    solve_ms = (time.time()-t0)*1000
    print("  lambda=%s: solved in %.0fms" % (lam, solve_ms))

    for dose in DOSES:
        dW_d = dW * dose
        with torch.no_grad():
            original = original_weight  # use pre-edit weight
            down_proj.weight.add_(dW_d.to(device=down_proj.weight.device, dtype=down_proj.weight.dtype))
            # Accuracy-based scoring on val set
            correct = 0
            for r, enc in zip(val_rows, val_encs):
                tokens = tokenizer(wcag_prompt(r["state"]), return_tensors="pt").input_ids.to(student.device)
                with torch.no_grad():
                    out = student(input_ids=tokens, use_cache=False)
                    logits = out.logits[0, enc, :]
                pred = logits[yes_id].item() > logits[no_id].item()
                correct += (pred == r["gold"]["wcag_violation"])
            down_proj.weight.copy_(original)
            val_acc = correct / len(val_rows)
            print("    dose=%.2f: val acc=%.1f%% (%d/%d)" % (dose, val_acc*100, correct, len(val_rows)))
            if val_acc > best_acc:
                best_acc = val_acc
                best = {"lam": lam, "dose": dose, "dW": dW_d.to(device=down_proj.weight.device, dtype=down_proj.weight.dtype), "val_acc": val_acc}

print("Best: lambda=%s, dose=%s, val_acc=%.1f%%" % (best["lam"], best["dose"], best["val_acc"]*100))

print("Applying edit...")
with torch.no_grad():
    down_proj.weight.add_(best["dW"])


In [ ]:
# Baseline (pre-edit) accuracy on test set
print("\n=== Baseline test accuracy (pre-edit) ===")
# Restore original weights temporarily
with torch.no_grad():
    saved_weight = down_proj.weight.clone()
    down_proj.weight.copy_(original_weight)  # from the last dose search
    correct = 0
    for r, enc in zip(test_rows, test_encs):
        tokens = tokenizer(wcag_prompt(r["state"]), return_tensors="pt").input_ids.to(student.device)
        with torch.no_grad():
            out = student(input_ids=tokens, use_cache=False)
        pred = out.logits[0, enc, yes_id].item() > out.logits[0, enc, no_id].item()
        correct += (pred == r["gold"]["wcag_violation"])
    baseline_acc = correct / len(test_rows)
    down_proj.weight.copy_(saved_weight)  # restore the edited weight
print("Baseline: %.1f%% (%d/%d)" % (baseline_acc*100, correct, len(test_rows)))

# Post-edit accuracy
print("\n=== Test accuracy (post-edit) ===")
correct = 0
for r, enc in zip(test_rows, test_encs):
    tokens = tokenizer(wcag_prompt(r["state"]), return_tensors="pt").input_ids.to(student.device)
    with torch.no_grad():
        out = student(input_ids=tokens, use_cache=False)
    pred = out.logits[0, enc, yes_id].item() > out.logits[0, enc, no_id].item()
    correct += (pred == r["gold"]["wcag_violation"])
test_acc = correct / len(test_rows) if test_rows else 0
print("Post-edit: %.1f%% (%d/%d)" % (test_acc*100, correct, len(test_rows)))
print("Improvement: %+.1f pp" % ((test_acc - baseline_acc) * 100))

# Save
os.makedirs("/kaggle/working/out_wcag", exist_ok=True)
student.save_pretrained("/kaggle/working/out_wcag")
tokenizer.save_pretrained("/kaggle/working/out_wcag")
meta = {"model": MODEL_NAME, "layer": LAYER, "width": WIDTH,
        "lambda": best["lam"], "dose": best["dose"],
        "val_acc": best["val_acc"], "test_acc": test_acc, "baseline_acc": baseline_acc,
        "n_fit": N_FIT, "n_val": N_VAL, "n_test": len(test_rows)}
with open("/kaggle/working/out_wcag/edit_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)
print("\nSaved to /kaggle/working/out_wcag")
print(json.dumps(meta, indent=2))
